# Project Infrastructure Layers (ArcPy)

**Purpose.** Project clipped infrastructure layers and calculate geometry attributes.

**Inputs.** Clipped infrastructure shapefiles.

**Outputs.** Projected infrastructure layers with length or area fields.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | ArcPy dependencies
import arcpy
import os
from pathlib import Path

arcpy.env.overwriteOutput = True

In [ ]:
# Cell | Projected-layer input and output workspaces
input_base = r"E:\Africa_Infrastructure_Data\infrastructure\clip"
output_base = r"E:\Africa_Infrastructure_Data\infrastructure\clip_project"

if not os.path.exists(output_base):
    os.makedirs(output_base)


In [ ]:
# Cell | Africa Albers Equal-Area coordinate reference system
# EPSG: 102022
# Use Africa Albers Equal Area to support comparable length and area measures.
target_sr = arcpy.SpatialReference(102022)


In [ ]:
# Cell | Feature-class geometry identification
def get_geometry_type(feature_class):
    """Return the ArcPy geometry type of a feature class."""
    desc = arcpy.Describe(feature_class)
    return desc.shapeType

In [ ]:
# Cell | Length and area attribute calculation
# Calculate length for polylines and area for polygon features.
def add_geometry_field(feature_class, geom_type):
    """Add and calculate length or area for the geometry type."""
    if geom_type == "Polyline":
        field_name = "Length"
        if field_name not in [f.name for f in arcpy.ListFields(feature_class)]:
            arcpy.AddField_management(feature_class, field_name, "DOUBLE")
        arcpy.CalculateField_management(feature_class, field_name, "!shape.length!", "PYTHON3")
        
    elif geom_type == "Polygon":
        field_name = "Area"
        if field_name not in [f.name for f in arcpy.ListFields(feature_class)]:
            arcpy.AddField_management(feature_class, field_name, "DOUBLE")
        arcpy.CalculateField_management(feature_class, field_name, "!shape.area!", "PYTHON3")


In [ ]:
# Cell | Projection and geometry-enrichment routine
def process_feature_class(input_fc, output_fc, target_sr):
    """Project a feature class and calculate geometry attributes."""
    try:
        arcpy.Project_management(input_fc, output_fc, target_sr)
        
        geom_type = get_geometry_type(output_fc)
        
        add_geometry_field(output_fc, geom_type)
        
        return True
    except Exception as e:
        return False

In [ ]:
# Cell | Annual input-folder discovery
year_folders = [f for f in os.listdir(input_base) if os.path.isdir(os.path.join(input_base, f))]

print("="*50)

In [ ]:
# Cell | Year-by-year infrastructure projection
success_count = 0
fail_count = 0

# Project every annual infrastructure layer using the same coordinate reference system.
for year_folder in year_folders:
    
    input_year_path = os.path.join(input_base, year_folder)
    output_year_path = os.path.join(output_base, year_folder)
    
    if not os.path.exists(output_year_path):
        os.makedirs(output_year_path)
    
    shapefiles = [f for f in os.listdir(input_year_path) if f.endswith('.shp')]
    
    
    for shp in shapefiles:
        input_fc = os.path.join(input_year_path, shp)
        output_fc = os.path.join(output_year_path, shp)
        
        
        if process_feature_class(input_fc, output_fc, target_sr):
            success_count += 1
        else:
            fail_count += 1